# Practical PyTorch · II · Part 7 — Companion Notebook

This goes with **"LoRA: Fine-Tune a Giant Model on a Free GPU"**. Run the cells top to bottom (Shift+Enter).

**Turn the GPU on first:** **Runtime → Change runtime type → Hardware accelerator → GPU**. LoRA makes big-model fine-tuning *possible* on a free GPU, but it still wants one.

## 0. Install the libraries

Colab has PyTorch already; we add Hugging Face's `transformers`, `peft` (parameter-efficient fine-tuning), and `datasets`.

In [ ]:
!pip install -q transformers peft datasets

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — turn it on: Runtime → Change runtime type → GPU")

## 1. Load a normal pretrained model

Nothing special yet — this is the same `from_pretrained` you've used all along. We'll use DistilBERT set up for 2-class sequence classification.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
base_model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

total = sum(p.numel() for p in base_model.parameters())
print(f"Base model has {total:,} parameters.")

## 2. Wrap it with LoRA

We freeze the whole model and bolt on tiny adapters. `LoraConfig` describes them; `get_peft_model` applies them.

- `r` = **size** of the adapters
- `lora_alpha` = **strength** of the adapters (convention: about 2×`r`)
- `target_modules` = **which layers** get adapters — these names are architecture-specific. `q_lin`/`v_lin` are DistilBERT's; Llama uses `q_proj`/`v_proj`, plain BERT uses `query`/`value`.
- `task_type` = `SEQ_CLS` for classification (`CAUSAL_LM` for a text-generating model).

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_lin", "v_lin"],  # DistilBERT-specific!
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(base_model, config)

## 3. The wow moment

Count how few parameters you're actually training. Everything else stays frozen, carrying its pretrained knowledge for free.

In [ ]:
model.print_trainable_parameters()
# trainable params: ~740K || all params: ~67M || trainable%: ~1.1

## 4. A small dataset to fine-tune on

We'll grab a slice of a sentiment dataset so this runs fast. (Swap in your own labelled text the same way.)

In [ ]:
from datasets import load_dataset

raw = load_dataset("imdb")
train_ds = raw["train"].shuffle(seed=42).select(range(1000))
eval_ds = raw["test"].shuffle(seed=42).select(range(500))

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_ds.map(tokenize, batched=True)
tokenized_eval = eval_ds.map(tokenize, batched=True)
print(tokenized_train)

## 5. Train it with the same `Trainer` as Part 5

This is the anticlimax in the best way: a LoRA-wrapped model trains exactly like any other Hugging Face model. The only nudge is a slightly higher learning rate — LoRA likes one.

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="lora-out",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    learning_rate=2e-4,
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
)

trainer.train()

## 6. Save the tiny adapter

`save_pretrained` on a PEFT model writes **only the adapter weights + a small config** — not the base model. Go look at the folder size and enjoy it.

In [ ]:
model.save_pretrained("my-adapter")

import os
size = sum(
    os.path.getsize(os.path.join("my-adapter", f)) for f in os.listdir("my-adapter")
)
print("Adapter folder contents:", os.listdir("my-adapter"))
print(f"Total adapter size: {size / 1e6:.2f} MB  (the base model was hundreds of MB)")

## 7. Load it back: base model + adapter

Loading is a two-step move, and the *why* is the whole swappability story: you load the original base model, then snap the adapter on top. Keep one base in memory and you can attach many different adapters to it.

In [ ]:
from peft import PeftModel

fresh_base = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
loaded = PeftModel.from_pretrained(fresh_base, "my-adapter")

# Quick sanity check: run it on a sentence.
loaded.eval()
inputs = tokenizer("This was an absolute delight from start to finish.", return_tensors="pt")
with torch.no_grad():
    logits = loaded(**inputs).logits
print("predicted label:", int(logits.argmax()))  # 1 = positive

That's the destination of Part 7: freeze a giant, train ~1% of it as adapters, save megabytes, and swap those adapters onto a shared base. Next up — **Part 8: When NOT to Fine-Tune** — the cheaper answers that beat fine-tuning more often than you'd think.